# Lab 03: Building a Simple AI Agent

In this lab you will build a basic AI agent that can use **tools** to
answer questions it couldn't handle with just its training data.

We'll use the `openai` library's function-calling feature to give the
model access to a calculator and a knowledge lookup tool.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["MODEL_ENDPOINT"],
    api_key=os.environ["MODEL_API_KEY"],
)

MODEL = os.environ["MODEL_NAME"]
print(f"Using model: {MODEL}")

## Step 1: Define tools

Tools are Python functions that the model can decide to call.
We describe them in a schema so the model knows what they do.

In [ ]:
def calculate(expression: str) -> str:
    """Evaluate a math expression and return the result."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def course_lookup(topic: str) -> str:
    """Look up information from the course knowledge base."""
    knowledge = {
        "office hours": "Professor Smith holds office hours Tue/Thu 2-4pm in Room 305.",
        "midterm": "The midterm exam is scheduled for October 15, covering chapters 1-6.",
        "project": "The final project is due December 10. Teams of 2-3 students.",
        "grading": "Grading: 30% homework, 25% midterm, 35% final project, 10% participation.",
    }
    for key, value in knowledge.items():
        if key in topic.lower():
            return value
    return f"No information found about '{topic}'. Try: office hours, midterm, project, grading."


available_tools = {
    "calculate": calculate,
    "course_lookup": course_lookup,
}

tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression, e.g. '2 + 2 * 3'"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "course_lookup",
            "description": "Look up course information like office hours, exam dates, or grading policy",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "Topic to look up"}
                },
                "required": ["topic"],
            },
        },
    },
]

print("Tools registered: calculate, course_lookup")

## Step 2: The agent loop

An agent works by:
1. Sending the user's question to the model along with tool descriptions
2. If the model wants to use a tool, we execute it and send the result back
3. The model incorporates the tool result into its final answer
4. Repeat until the model gives a direct response

In [ ]:
def run_agent(question: str, verbose: bool = True) -> str:
    """Run the agent loop for a given question."""
    messages = [
        {"role": "system", "content": "You are a helpful course assistant. Use the available tools when needed to answer accurately."},
        {"role": "user", "content": question},
    ]

    for step in range(5):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tool_schemas,
        )

        choice = response.choices[0]

        if choice.finish_reason == "tool_calls" and choice.message.tool_calls:
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                if verbose:
                    print(f"  [Step {step+1}] Calling {fn_name}({fn_args})")

                result = available_tools[fn_name](**fn_args)
                if verbose:
                    print(f"           -> {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
        else:
            return choice.message.content

    return "Agent reached maximum steps without a final answer."


print("Agent ready. Try run_agent('your question here')")

## Step 3: Try it out

In [ ]:
print(run_agent("What's 15% of 2400?"))

In [ ]:
print(run_agent("When is the midterm and what does it cover?"))

In [ ]:
print(run_agent("If I get 85% on the midterm and 90% on the final project, what would my weighted score be for those two components?"))

## Exercise

1. Add a third tool -- for example, a `weather` function or a `dictionary` lookup
2. Ask the agent a question that requires using multiple tools in sequence
3. What happens if you ask something none of the tools can answer?

In [ ]:
# Your code here
